# Google search engine

## Contenido:

* [Librerias](#Librerias)
* [Base de datos](#Datos)
* [Escobar el patron del mal](#Escobar)
* [API JSON de GCP](#Api)
* [Motor de busqueda](#Motor)

<a name="Librerias"></a>
## Librerias

In [ ]:
!pip install openpyxl==3.0.9
!pip install beautifulsoup4
!pip install google
import pandas as pd
import requests
import csv
import time
from openpyxl import Workbook

from google.colab import drive
drive.mount('/content/drive/')

from datetime import datetime
from datetime import date

import os.path
from os import path

PATH_TO_THESAURUS = 'https://drive.google.com/uc?export=download&id=19fNhQW8my4mToBVvKB9-8rRJFsEqN_Y0'

<a name="Datos"></a>
##  Bases de datos (Palabras Claves y Criterios)

### Importando Datos

In [ ]:
palabras_clave_dataframe = pd.read_excel(PATH_TO_THESAURUS, sheet_name = 'PalabrasClave')
search_params_dataframe = pd.read_excel(PATH_TO_THESAURUS, sheet_name = 'search_params')
palabras_clave_dataframe

<a name="Escobar"></a>
### Escobar, El Patrón del mal OJO => (Faltan las otras series.)

In [ ]:
escobar_el_patron_dataframe = palabras_clave_dataframe.loc[palabras_clave_dataframe['Nombres de las telenovelas y  Series'] == 'Escobar, El Patrón del mal']

lista_contendios_historicos = escobar_el_patron_dataframe['Palabras Clave Contenidos Históricos'].dropna().unique().tolist()
lista_valoraciones_generales = escobar_el_patron_dataframe['Palabras clave Valoración General'].dropna().tolist()

lista_websites = search_params_dataframe['URL'].dropna().tolist()
lista_paises = search_params_dataframe['Pais'].dropna().tolist()

In [ ]:
escobar_el_patron_dataframe

In [ ]:
lista_contendios_historicos

In [ ]:
lista_valoraciones_generales

In [ ]:
Criterios=[]
for i in lista_contendios_historicos:
  for j in lista_valoraciones_generales:
    x=i+' '+j
    Criterios.append(x)


In [ ]:
print(len(Criterios))
Criterios

In [ ]:
lista_websites

In [ ]:
lista_paises

<a name="Api"></a>
## Programmable Search Engine (Custom Search JSON API de GCP)

### Ejecucion

#### Parametros constantes

In [ ]:
#fields = 'searchInformation,queries(request(count,cr,exactTerms,filter,gl,hl,language,orTerms,searchTerms,siteSearch,siteSearchFilter,sort,startIndex,title))\
#,items(cacheId,title,link,snippet,pagemap(metatags(article:published_time)))'
fields=''
URI = 'https://www.googleapis.com/customsearch/v1?key='
key = 'TU_CLAVE_DE_API' # Clave del motor de busqueda
cx = 'f766e939438207fda' # ID del motor de busqueda

# =================== OJO => (Modificar para otras series) ===================

exactTerms = 'Escobar, el patrón del mal' # Frase que deben contener todos los documentos de los resultados de búsqueda OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

filter = '1' # Controla la activación o desactivación del filtro de contenido duplicado 1: Activa el filtro de contenido duplicado.
hl = 'es' # Establece el idioma de la interfaz de usuario.
lr = 'lang_es' # Lenguaje
num = '10' # Numero de resultados de busqueda

# =================== OJO => (Modificar para otras series) ===================

orTerms = 'Serie' #Proporciona términos de búsqueda adicionales para verificar en un documento OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

siteSearchFilter = 'e' # Controla si incluir o excluir resultados 

#### Parametros variables

In [ ]:
# =================== OJO => (Modificar para otras series) ===================

cr = ['countryCO','countryBR','countryCL','countryDE', 'countryUS'] # Restringe los resultados de la búsqueda a los documentos que se originan en un país en particular. OJO => (Revisar)

# =================== OJO => (Modificar para otras series) ===================

# =================== OJO => (Modificar para otras series) ===================

gl = ['co','br','cl','de', 'us'] # Geolocalización de usuario final br brasil, co colombia, cl chile, de Alemania, us EEUU OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

Pais={'co':'Colombia','br':'Brasil','cl':'Chile','de':'Alemania','us':'Estados Unidos'} # Nombre de paises deacuerdo a su codigo


# hq = #Agrega los términos de consulta especificados a la consulta, como si estuvieran combinados con un operador lógico AND.

# =================== OJO => (Modificar para otras series) ===================

q = 'Escobar, el Patrón del mal Colombia historia' # Consulta  OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

siteSearch ='https://youtube.com/' # Especifica un sitio determinado que siempre debe incluirse o excluirse de los resultados

# =================== OJO => (Modificar para otras series) ===================

sort = ['date:r:20090101:20130101','date:r:20130101:20140101','date:r:20140101:20150101'\
        ,'date:r:20150101:20160101','date:r:20160101:20170101','date:r:20170101:20180101'\
        ,'date:r:20180101:20190101','date:r:20190101:20200101','date:r:20200101:20210101'] # La expresión de clasificación que se aplicará a los resultados. OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

# =================== OJO => (Modificar para otras series) ===================

fecha = {'date:r:20090101:20130101':'20090101-20130101','date:r:20130101:20140101':'20130101-20140101','date:r:20140101:20150101':'20140101-20150101'\
         ,'date:r:20150101:20160101':'20150101-20160101','date:r:20160101:20170101':'20160101-20170101','date:r:20170101:20180101':'20170101-20180101'\
         ,'date:r:20180101:20190101':'20180101-20190101','date:r:20190101:20200101':'20190101-20200101','date:r:20200101:20210101':'20200101-20210101'} # Rango de fechas deacuerdo a su codigo

# =================== OJO => (Modificar para otras series) ===================

start = '1' # El índice del primer resultado que se devuelve.

#### Solicitud

In [ ]:
r = requests.get(URI+key+'&cx='+cx+'&cr='+cr[0]+'&exactTerms='+exactTerms+'&filter='+filter+'&fields='+fields\
                 +'&gl='+gl[0]+'&hl='+hl+'&lr='+lr+'&num='+num+'&orTerms='+orTerms+'&q='+q\
                 +'&siteSearch='+siteSearch+'&siteSearchFilter='+siteSearchFilter+'&sort='+sort[0]+'&start='+start) 

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d

In [ ]:
d.keys()

In [ ]:
d['kind']

In [ ]:
d['url']

In [ ]:
d['queries']

In [ ]:
d['context']

In [ ]:
d['searchInformation']

In [ ]:
metadata1 = pd.json_normalize(d['queries']['request'])
metadata2 = pd.json_normalize(d['searchInformation'])
metadata = pd.concat([metadata1,metadata2],axis=1)
metadata.info()

In [ ]:
metadata

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data = pd.json_normalize(items)
data.info()

In [ ]:
data

In [ ]:
columnas=list(data.columns)
infodatac=pd.DataFrame()
for i in range(len(data)):
  if ('pagemap.metatags' in  columnas):
    if (str(data['pagemap.metatags'][i])!='nan'):
      infodata = pd.json_normalize(data['pagemap.metatags'][i])
      infodatac=pd.concat([infodatac,infodata],axis=0)
    else:
      infodata = pd.json_normalize([{'Texto':'No hay pagemap.metatags'}])
      infodatac=pd.concat([infodatac,infodata],axis=0)
  else:
    infodata = pd.json_normalize([{'Texto':'No hay pagemap.metatags'}])
    infodatac=pd.concat([infodatac,infodata],axis=0)


In [ ]:
infodatac.info()

In [ ]:
infodatac

In [ ]:
infodatac=infodatac.reset_index()

In [ ]:
dataglobal = pd.concat([data,infodatac],axis=1)
dataglobal.info()

In [ ]:
dataglobal

<a name="Motor"></a>
### Motor de Busqueda en produccion

#### Parametros fijos

In [ ]:
fields = 'searchInformation,queries(request(count,cr,exactTerms,filter,gl,hl,language,orTerms,searchTerms,siteSearch,siteSearchFilter,sort,startIndex,title))\
,items(cacheId,title,link,snippet,pagemap(metatags(article:published_time)))'
URI = 'https://www.googleapis.com/customsearch/v1?key='
key = 'TU_CLAVE_DE_API' # Clave del motor de busqueda
cx = 'f766e939438207fda' # ID del motor de busqueda 

# =================== OJO => (Modificar para otras series) ===================

Telenovela = 'Escobar, el patrón del mal ' # Telenovela  OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

# =================== OJO => (Modificar para otras series) ===================

exactTerms = 'Escobar, el patrón del mal' # Frase que deben contener todos los documentos de los resultados de búsqueda OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

filter = '1' # Controla la activación o desactivación del filtro de contenido duplicado 1: Activa el filtro de contenido duplicado.
hl = 'es' # Establece el idioma de la interfaz de usuario.
lr = 'lang_es' # Lenguaje
num = '10' # Numero de resultados de busqueda

# =================== OJO => (Modificar para otras series) ===================

orTerms = 'Serie' # Proporciona términos de búsqueda adicionales para verificar en un documento OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

siteSearchFilter = 'e' # Controla si incluir o excluir resultados 

#### Parametros variables

In [ ]:
# =================== OJO => (Modificar para otras series) ===================

cr = ['countryCO','countryBR','countryCL','countryDE', 'countryUS'] # Restringe los resultados de la búsqueda a los documentos que se originan en un país en particular. OJO => (Revisar)

# =================== OJO => (Modificar para otras series) ===================

# =================== OJO => (Modificar para otras series) ===================

gl = ['co','br','cl','de', 'us'] # Geolocalización de usuario final br brasil, co colombia, cl chile, de Alemania, us EEUU OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

Pais={'co':'Colombia','br':'Brasil','cl':'Chile','de':'Alemania','us':'Estados Unidos'} # Nombre de paises deacuerdo a su codigo


# hq = #Agrega los términos de consulta especificados a la consulta, como si estuvieran combinados con un operador lógico AND.

# =================== OJO => (Modificar para otras series) ===================

q = 'Escobar, el Patrón del mal Colombia historia' # Consulta  OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

siteSearch ='https://youtube.com/' # Especifica un sitio determinado que siempre debe incluirse o excluirse de los resultados

# =================== OJO => (Modificar para otras series) ===================

sort = ['date:r:20090101:20130101','date:r:20130101:20140101','date:r:20140101:20150101'\
        ,'date:r:20150101:20160101','date:r:20160101:20170101','date:r:20170101:20180101'\
        ,'date:r:20180101:20190101','date:r:20190101:20200101','date:r:20200101:20210101'] # La expresión de clasificación que se aplicará a los resultados. OJO => (Modificar para otras series)

# =================== OJO => (Modificar para otras series) ===================

# =================== OJO => (Modificar para otras series) ===================

fecha = {'date:r:20090101:20130101':'20090101-20130101','date:r:20130101:20140101':'20130101-20140101','date:r:20140101:20150101':'20140101-20150101'\
         ,'date:r:20150101:20160101':'20150101-20160101','date:r:20160101:20170101':'20160101-20170101','date:r:20170101:20180101':'20170101-20180101'\
         ,'date:r:20180101:20190101':'20180101-20190101','date:r:20190101:20200101':'20190101-20200101','date:r:20200101:20210101':'20200101-20210101'} # Rango de fechas deacuerdo a su codigo

# =================== OJO => (Modificar para otras series) ===================

start = '1' # El índice del primer resultado que se devuelve.

#### Carpetas Drive 

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/GoogleResults/SearchResults/EscobarPatrondelMal/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'SEARCHDATOS'+str(datetime.now())

#### Constantes

In [ ]:
# ======================= OJO Cambiar diariamente =======================

ith_Criterio=498 # (Criterio = contenido historico + valoracion general) donde va comenzar la busqueda 

# ======================= OJO Cambiar diariamente =======================

solicitud = 0 # Inicio de las solicitudes
solicitudmax = 9000 # Numero de solicitudes maximas (el limite es 100 de manera gratuita o 10000 con costo diario) #valor para detener la ejecucion bien sea por Criterios o por solicitudes.
espera = 1 # Valor de retardo para las solicitudes
START = [1,10,20,30,40,50,60,70,80,90] # Parametro donde comienza la paginacion (El numero maximo es 100 y start + num no puede ser mayor de 100)

#### Ejecucion

In [ ]:
Datac = pd.DataFrame() # Dataframe de Resultados Totales
Metadatac = pd.DataFrame() # Dataframe de Metadatos de Resultados Totales
Nosolicitudes = pd.DataFrame(columns=['Solicitud','Telenovela','CriterioNumero','Criterio'\
                                      ,'Consulta','Pais','Año2009_2021','Paginacion','FechaEjecucion']) # Dataframe con datos correspondientes a solicitudes malas.

for Criterio in Criterios[ith_Criterio:]: # Lista de Criterios

  CriterioNumero = Criterios.index(Criterio) # Numero de Criterio
  
  if (solicitud == solicitudmax + 1): 
    break
  
  q = Telenovela + Criterio # Consulta o query

  for ith_pais in range(len(gl)): # Lista de paises
    
    if (solicitud == solicitudmax + 1):
      break
    
    for ith_fecha in sort[:]: # Lista de Fechas
      
      if (solicitud == solicitudmax + 1):
        break
      
      for index_pagina in START[:]: # Lista de indices iniciales de paginacion
        
        solicitud = solicitud + 1

        if (solicitud == solicitudmax + 1):
          ptoPartida = [CriterioNumero, Criterio, gl[ith_pais], ith_fecha, index_pagina] # Punto de partida para volver a comenza r el motor de busuqeda en produccion.
          break

        request = URI+key+'&cx='+cx+'&cr='+cr[ith_pais]+'&exactTerms='+exactTerms+'&filter='+filter+'&fields='+fields\
        +'&gl='+gl[ith_pais]+'&hl='+hl+'&lr='+lr+'&num='+num+'&orTerms='+orTerms+'&q='+q\
        +'&siteSearch='+siteSearch+'&siteSearchFilter='+siteSearchFilter+'&sort='+ith_fecha+'&start='+str(index_pagina)
        
        time.sleep(espera) # Tiempo de espera por cada solicitud
        r = requests.get(request) # Solicitud
        print('solicitud',solicitud,r.reason) # Respuesta

        if (r.reason == 'OK'): # Vemos si la solicitud fue exitosa
          d = dict(r.json())
          items = d.get('items')
          Metadata1 = pd.json_normalize(d['queries']['request'])
          Metadata2 = pd.json_normalize(d['searchInformation'])
          Metadata = pd.concat([Metadata1,Metadata2],axis = 1).assign(Telenovela = Telenovela, Consulta = q, CriterioBusqueda = Criterio\
                                                                      , CriterioNumero = CriterioNumero, FechaBusqueda = today\
                                                                      , Solicitud = solicitud)
          
          if items!=None: # Vemos si la solicitud exitosa tiene resultados
            Data = pd.json_normalize(items).assign(Telenovela = Telenovela, Consulta = q, CriterioBusqueda = Criterio, Criterio = 'C' + str(CriterioNumero)\
                                                   , Pais = Pais[gl[ith_pais]], Año2009_2021 = fecha[ith_fecha], Solicitud = solicitud)
            
            # ==Normalizar JSON en columna pagemap.metatags==
            columnas = list(Data.columns)
            infodatac = pd.DataFrame()
            for i in range(len(Data)):
              if ('pagemap.metatags' in  columnas):
                if (str(Data['pagemap.metatags'][i]) != 'nan'):
                  infodata = pd.json_normalize(Data['pagemap.metatags'][i])
                  infodatac = pd.concat([infodatac, infodata], axis = 0)
                else:
                  infodata = pd.json_normalize([{'Texto':'No hay pagemap.metatags'}])
                  infodatac = pd.concat([infodatac, infodata], axis = 0)
              else:
                infodata = pd.json_normalize([{'Texto':'No hay pagemap.metatags'}])
                infodatac = pd.concat([infodatac,infodata], axis = 0)
            infodatac = infodatac.reset_index()
            # ==Normalizar JSON en columna pagemap.metatags==
            
            Dataglobal = pd.concat([Data, infodatac],  axis = 1)
            Datac = pd.concat([Datac, Dataglobal], ignore_index = True) # Dataframe de Resultados Totales
            Metadata = Metadata.assign(Resultados = 'Hay Resultados')
            Metadatac = pd.concat([Metadatac, Metadata], ignore_index = True) # Dataframe de Metadatos donde no hay resultados
          
          else: # Vemos si la solicitud exitosa no tiene resultados
            Metadata = Metadata.assign(Resultados = 'No hay Resultados')
            Metadatac = pd.concat([Metadatac, Metadata], ignore_index = True) # Dataframe de Metadatos de Resultados Totales
            print("No hay links para Solicitud ",solicitud)
            break
            
        else: # Vemos si la solicitud no fue exitosa
          Norequest = [solicitud,Telenovela,CriterioNumero,Criterio,q,gl[ith_pais],ith_fecha,index_pagina,today]
          Nosolicitudes = Nosolicitudes.append({'Solicitud':solicitud, 'Telenovela':Telenovela, 'CriterioNumero':CriterioNumero\
                                                , 'Criterio':Criterio, 'Consulta':q, 'Pais':gl[ith_pais], 'Año2009_2021':ith_fecha\
                                                , 'Paginacion':index_pagina, 'FechaEjecucion':today},ignore_index = True)
          print("No se pudo realizadar la solicitud ",solicitud)

print("Termino la busqueda")

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

In [ ]:
Nosolicitudes

In [ ]:
ptoPartida

#### Estructurar y dar formato a Dataframe

In [ ]:
Datacd=Datac.drop(['pagemap.metatags', 'index', 'Texto'], axis=1)
Datacd.info()

In [ ]:
Datacd

In [ ]:
columnas={'title':'Titulo','link':'SitiosWeb/Fuente','snippet':'Resumen'\
          ,'cacheId':'Id','Telenovela':'Telenovela','Consulta':'Consulta'\
          ,'CriterioBusqueda':'CriteriodeBusqueda','Criterio':'HallazgoxCriterio','Pais':'Pais'\
          ,'Año2009_2021':'Año2009-2021','Solicitud':'Solicitud','article:published_time':'FechaPublicacion'}

columnasorden=['FechaPublicacion', 'Pais', 'Telenovela', 'Consulta', 'SitiosWeb/Fuente', 'CriteriodeBusqueda'\
               , 'Resumen', 'Año2009-2021', 'HallazgoxCriterio', 'Titulo', 'Id', 'Solicitud']

Datacd.rename(columns=columnas,inplace=True)
DATA=Datacd.reindex(columns=columnasorden)
DATA

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
Nosolicitudes.to_csv(PATH_TO_DIR_0+'/'+'NOSOLICITUDES-'+file_nameL+'.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)